# **Resume Screening Application Using NLP**

## Download dataset from Kaggle


In [ ]:
from google.colab import files
files.upload() #this will prompt you to upload the kaggle.json

In [ ]:
!ls -lha kaggle.json

In [ ]:
!pip install -q kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!pwd

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets download -d gauravduttakiit/resume-dataset

In [ ]:
!unzip resume-dataset.zip

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import re

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.multiclass import OneVsRestClassifier

nltk.download('punkt_tab')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
resume_data = pd.read_csv('UpdatedResumeDataSet.csv')
resume_data.head()

## Data Exploration

In [ ]:
resume_data.shape

In [ ]:
resume_data.info()

In [ ]:
resume_data.describe()

In [ ]:
counts = resume_data["Category"].value_counts()
labels = resume_data["Category"].unique()

plt.figure(figsize=(13, 8))
plt.pie(counts, labels=labels, autopct='%1.f%%', colors=sns.color_palette("Spectral"))
plt.title('Resume Category Distribution')
plt.show()

In [ ]:
# plt.xticks(rotation=90)
sns.countplot(y=resume_data["Category"], width=0.8)
plt.show()

## Cleaning Data and Feature Engineering

In [ ]:
def cleanResume(resumeText):
    resumeText = re.sub('http\S+\s*', ' ', resumeText)  # remove URLs
    resumeText = re.sub('RT|cc', ' ', resumeText)  # remove RT and cc
    resumeText = re.sub('#\S+', '', resumeText)  # remove hashtags
    resumeText = re.sub('@\S+', '  ', resumeText)  # remove mentions
    resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', resumeText)  # remove punctuations
    resumeText = re.sub(r'[^\x00-\x7f]',r' ', resumeText) # remove special characters
    resumeText = re.sub('\s+', ' ', resumeText)  # remove extra whitespace
    resumeText = resumeText.lower()  # convert to lowercase

    word_tokens = nltk.word_tokenize(resumeText)
    stop_words = set(stopwords.words('english'))
    filtered_text = [word for word in word_tokens if word not in stop_words]
    resumeText = ' '.join(filtered_text)

    return resumeText

In [ ]:
resume_data['Resume'] = resume_data.Resume.apply(lambda x: cleanResume(x))
resume_data.head()

In [ ]:
labelEncoder = LabelEncoder()
resume_data["Category"] = labelEncoder.fit_transform(resume_data["Category"])

In [ ]:
print(sorted(resume_data.Category.unique()))

{6: 'Data Science',
 12: 'HR',
 0: 'Advocate',
 1: 'Arts',
 24: 'Web Designing',
 16: 'Mechanical Engineer',
 22: 'Sales',
 14: 'Health and fitness',
 5: 'Civil Engineer',
 15: 'Java Developer',
 4: 'Business Analyst',
 21: 'SAP Developer',
 2: 'Automation Testing',
 11: 'Electrical Engineering',
 18: 'Operations Manager',
 20: 'Python Developer',
 8: 'DevOps Engineer',
 17: 'Network Security Engineer',
 19: 'PMO',
 7: 'Database',
 13: 'Hadoop',
 10: 'ETL Developer',
 9: 'DotNet Developer',
 3: 'Blockchain',
 23: 'Testing'}

In [ ]:
resume_data.head()

## Vectorization

In [ ]:
tfidf = TfidfVectorizer(stop_words="english",)
X = tfidf.fit_transform(resume_data["Resume"]).toarray()
y = resume_data["Category"]

## Data Splitting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=41)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

## Model Creation and Prediction

In [ ]:
model = OneVsRestClassifier(KNeighborsClassifier(n_neighbors = 23, weights = 'distance'))
model.fit(X_train,y_train)
y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

## Prediction System

In [ ]:
import pickle
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))

In [ ]:
pred_model = pickle.load(open("model.pkl", "rb"))
pred_tfidf = pickle.load(open("tfidf.pkl", "rb"))

In [ ]:
category_map = {
    6: 'Data Science',
    12: 'HR',
    0: 'Advocate',
    1: 'Arts',
    24: 'Web Designing',
    16: 'Mechanical Engineer',
    22: 'Sales',
    14: 'Health and fitness',
    5: 'Civil Engineer',
    15: 'Java Developer',
    4: 'Business Analyst',
    21: 'SAP Developer',
    2: 'Automation Testing',
    11: 'Electrical Engineering',
    18: 'Operations Manager',
    20: 'Python Developer',
    8: 'DevOps Engineer',
    17: 'Network Security Engineer',
    19: 'PMO',
    7: 'Database',
    13: 'Hadoop',
    10: 'ETL Developer',
    9: 'DotNet Developer',
    3: 'Blockchain',
    23: 'Testing'
}

In [ ]:
elect_eng_resume = """
BOB ROSS
example@example.com | (555) 555-5555 | Atlanta, GA 31139

SUMMARY STATEMENT

Dedicated, professional and highly experienced electrical engineer who has been in the field for nearly 20
years. Proficient in a wide variety of engineering software. Excellent communication and time
management skills. Reliable and strives to go above and beyond to deliver a project that meets and
exceeds clients' expectations. Excellent team player.

CORE QUALIFICATIONS

. Equipment design
. Schematic development
. Quality assurance
. Prototype development

EDUCATION

Master of Science: Electrical & Computer Engineering
Georgia Institute of Technology | Atlanta, GA
Bachelor of Science: Electrical Engineering
Georgia Institute of Technology | Atlanta, GA

WORK EXPERIENCE

Nov 2012 - Current
Electrical Engineer
Leidos - Atlanta, GA

+ 3D modeling
. Debugging and troubleshooting
. Time management
. Teamwork

Sep 2006 - Oct 2012
Design Engineer
Emonics, LLC - Atlanta, GA

Jun 2002 - Aug 2006
Electrical Project Engineer
EYP - Atlanta, GA

. Perform engineering tasks using computer-assisted engineering
methods and design software.
. Communicate with customers, other engineers and other
relevant parties to ensure current engineering projects are on
task and to finalize details of upcoming projects.
. Coordinate the manufacture, installation, maintenance and
support of electrical engineering projects to ensure safety and
compliance with customer requirements and local, state and
federal laws.
. Supervised a team of 20 electrical engineers on multi-million
dollar projects throughout the state.
. Performed detailed calculations to ensure that projects were
manufactured, constructed and installed within specific
standards and guidelines, reducing risks by 95%.
. Created budgets for three state-wide projects that included
materials, construction costs and labor costs.
. Presented the budgets to the proper parties and adjusted them
where needed.
. Studied maps, conducted field surveys and reviewed other data
to identify and repair problems in power systems.
. Maintained electrical equipment and instruments to ensure
safety and working equipment for each project.
. Oversaw 10 projects to assure they were completed on time and
within the proper budget.
. Designed electrical systems meant to work in connection with
natural lighting or in other ways to minimize the requirement
for electrical energy.
. Developed project programs to secure new equipment and
perform major repairs when necessary, improving productivity

"""

cleaned_elect_eng_resume = cleanResume(elect_eng_resume)

input_features_el = pred_tfidf.transform([cleaned_elect_eng_resume]).toarray()
prediction_el = pred_model.predict(input_features_el)

print(category_map[int(prediction_el)])

In [ ]:
automation_test_resume="""
First Last
Automation Tester
Quality-focused auto metion tester with & years of excerience in web end mobile app
tessling. Praven abilily la fnd and isalale delees through erengreherisive exp nalory
testing. Key achieve men :: Overhauled !: Ul automated testing solutions lead ng to a
draralin 89% increase in leedt lraceabilily ard oveall praduet qualily

WORK EXPERIENCE

CONTACT

. Denver, OH (Open to Remote)
: +1-223-912-009
. mral@resewerded.comn
. linkedn.com/In/username

5KILLS

Automation Tester
Resume Worded, New York, NY
. Automated manual test cases using Selenium WebDriver and Python
to improve the efficiency of testing by 92%
. Reporled and tracked software cefects usrig bug-tracking toos arid
assisted with root cause anslysis, resolved 75% of issues d'soovered.
. Overhauled 5 Ul automated testing solutions eading to a dramatic
89% increase in test traceability and overall product qua ity.
. Tested 15 versions cf RW's software taking account of unique
geographical terrain and 10 browser p atferms, achieved an 85%
product uscability ratc.

Information Security Analyst
Growthsi, San Francisco, CA

. Analyzed system logs, network traffic, and 15 other cigital footprints
to detect unauthonzed access attempts, prevented 2K mal cicus
attacks in Q1 2014 alone.
. Created an automated tool that allowed over 10K users to submit
requesls for access righls changes via a web purlal, reducing the use
of emailing Word documents by 95% belween 18 depar lments.
. Implemented two factor autheritication on 32 orit cal systems usirg
hardware tokens, preventing 1.5K unauthorized acess requests to
sensitive databases.
. Conducted risk assessments on 20 company assets, including
software and harcware systems, identif ed and resoived 2K potential
sccurity vulnerabilitics in 3 menths.

Data Scientist
Resume Worded's Exciting Company, New York, NY
. Desigried an algorithm that leveraged natural language processi"g
(NLP) techniques to extract structured informat on from unstructured
medical records in real time, resulting in a SX reduction in the time
needed to process 85K patients' data curng onboarcing
Buit a recommencer system using co laborative filterng a gorithms
to recemmend personalized care plans for 29 patients based on their
discase state and treatment history, increasing physisian efficieney by
55%
. Implemented a machine learning framework capable of detecting
anomalles with n large volumes of streaming loT sensor data at
millisecond resolutien, recucing downtime acmss power grids, oil
and gas pipclines by up to 75%.

November 2015 - Present

June 2013 - October 2015

iechical
. Belley MicroStation (Advance:)
· Autodesk AutoCAD (Experienced)
. Vware
. Puppet
. Vsible Razor

Tectviiques
. Regression testng
. MargoDB
. AWS Cloud Sarvices

Toalg

. JUnr
· Selenium
· DEbugview
. RegMon

. Don't forget to use Beeume
Warrlod to soan your resune
belore you send it off (it's free and
proven to get you more joos)

August 2010 - January 2013

.

EDUCATION

Resume Worded University
Bachelor of Science
Appled Matherratics
Bosiun, MA - May 2009
Awrds: Resmne Worded Tnaching
Fellow (enly 5 ewarded to class). Dean's
List 2012 (Top 10%)

OTHER

. Softwere Testng Clus.
. Merribes, Soll wars Quelily
Assurance Horums."""

cleaned_automation_test_resume = cleanResume(automation_test_resume)

input_features_at = pred_tfidf.transform([cleaned_automation_test_resume]).toarray()
prediction_at = pred_model.predict(input_features_at)
print(category_map[int(prediction_at)])

In [ ]:
hadoop_resume = """
First Last
Hadoop Developer
Grand Forks, North Dakota . +1-234-456-789 . professionalemail@resumeworded.com . linkedin.com/in/usemame

Hadoop developer with 8+ years of experience in hamnessing the power of big data to drive actionable insights and
optimize data processing workflows. Key achievement: Collaborated with 100+ data scientists to create machine
learning models, leveraging big data for predictive analytics.

RELEVANT WORK EXPERIENCE

Resume Worded, New York, NY
Hadoop Developer
. Boosted ETL processes, reducing data processing time by 82% and improving overall system efficiency
within three months of assuming the role.
. Implemented data validation checks, resulting in a 56% decrease in errors and enhancing data reliability in
the first year.
. Managed Hadoop clusters to achieve 99.9% uptime and ensure scalability to handle 220+ terabytes of data.
. Optimized MapReduce jobs, leading to a 77% improvement in query performance for complex analytical
tasks

Growthsi, San Francisco, CA
Linux Administrator
. Implemented a backup strategy that reduced data loss risk by 78% and improved recovery time by 24
hours.
. Strengthened system security by implementing SELinux policies, reducing security incidents by 15% in 2014
and 2015.
. Managed user accounts and permissions by streamlining the onboarding process and reducing
access-related issues by 34%.
. Developed shell scripts, automating system monitoring, backups, and log analysis, saving 21 person-hours
monthly.

Resume Worded Exciting Company, San Francisco, CA
Scala Developer
. Developed a highly scalable backend system handling 230+ concurrent users, leading to a 44% increase in
user satisfaction.
. Refactored legacy Java codebase into Scala, reducing codebase size by 87% and enhancing maintainability
in the first year.
. Designed and implemented RESTful APIs, leading to a 50% reduction in integration time for 10+ third-party
services.

EDUCATION

Resume Worded University, New York, NY
Associate of Science - Information Technology

SKILLS

2015 - Present

2013-2015

2011-2013

2011

Technical Skills: Apache NiFi (Advanced), Talend (Experienced), Linux/Unix Shell Scripting, Power BI, GIT
Languages: English (Native), German (Fluent), French (Conversational)
"""

cleaned_hadoop_resume = cleanResume(hadoop_resume)

input_features_hadoop = pred_tfidf.transform([cleaned_hadoop_resume]).toarray()
prediction_hadoop = pred_model.predict(input_features_hadoop)
print(category_map[int(prediction_hadoop)])